# Bronze ingestion: Elhub consumption data

In [1]:
from pathlib import Path
import hashlib
import requests
import json
import re
import tempfile
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import unquote, urlparse
from uuid import uuid4

# Root directory of the project is two levels up from this script
PROJECT_ROOT = Path.cwd().resolve().parent.parent

# Dataset and source URL
DATASET = "CONSUMPTION_PER_GROUP_MBA_HOUR"
SOURCE_URL = f"https://api.elhub.no/energy-data/v0/price-areas.csv?dataset={DATASET}"

# Create the bronze directory for storing the downloaded data
BRONZE_DIR = PROJECT_ROOT / "data" / "bronze" / "elhub" / DATASET
BRONZE_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Bronze directory:", BRONZE_DIR)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find project root")

Project root: /home/roland/Desktop/DAT535_DEV2/group14
Bronze directory: /home/roland/Desktop/DAT535_DEV2/group14/data/bronze/elhub/CONSUMPTION_PER_GROUP_MBA_HOUR


In [2]:
# Dataset configuration
DATASET = "CONSUMPTION_PER_GROUP_MBA_HOUR"
SOURCE_URL = f"https://api.elhub.no/energy-data/v0/price-areas.csv?dataset={DATASET}"

# Save data in project-root/data/bronze/elhub/
BRONZE_DIR = PROJECT_ROOT / "data" / "bronze" / "elhub" / DATASET
BRONZE_DIR.mkdir(parents=True, exist_ok=True)

# Run information
run_id = uuid4().hex[:12]
started_at = datetime.now(timezone.utc).isoformat()

print("Project root:", PROJECT_ROOT)
print("Dataset:", DATASET)
print("Source:", SOURCE_URL)
print("Bronze folder:", BRONZE_DIR)
print("Run ID:", run_id)

Project root: /home/roland/Desktop/DAT535_DEV2/group14
Dataset: CONSUMPTION_PER_GROUP_MBA_HOUR
Source: https://api.elhub.no/energy-data/v0/price-areas.csv?dataset=CONSUMPTION_PER_GROUP_MBA_HOUR
Bronze folder: /home/roland/Desktop/DAT535_DEV2/group14/data/bronze/elhub/CONSUMPTION_PER_GROUP_MBA_HOUR
Run ID: 98eb462bce0f


## 2. Download the raw ZIP file

In [3]:
print("Downloading from Elhub...")

# A temporary file is created inside the Bronze folder.
with tempfile.NamedTemporaryFile(
    dir=BRONZE_DIR, prefix="download_", suffix=".tmp", delete=False
) as temp_file:
    temp_path = Path(temp_file.name)

try:
    with requests.get(SOURCE_URL, stream=True, timeout=(15, 180)) as response:
        response.raise_for_status()  # Stop if the server returned an HTTP error.
        final_url = response.url
        total_bytes = 0
        with temp_path.open("wb") as output:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    output.write(chunk)
                    total_bytes += len(chunk)
                    print(f"  Received {total_bytes / 1024 / 1024:.1f} MiB", end="\r")

    print(f"\nDownload finished: {total_bytes:,} bytes")
    print("Redirected URL:", final_url)
except Exception:
    temp_path.unlink(missing_ok=True)
    raise

if total_bytes == 0:
    temp_path.unlink(missing_ok=True)
    raise ValueError("Downloaded file is empty")

  Received 10.8 MiB
Download finished: 11,313,160 bytes
Redirected URL: https://frzq0sxltynr.objectstorage.eu-frankfurt-1.oci.customer-oci.com/p/hsnBS_rZZzT8AyoMxky1hRkvgrGCggmqDoCPntmrmKrtTrJfmkvpQpRgtct_3UBW/n/frzq0sxltynr/b/elhub-prod1-dataflow-dataset-osb/o/CONSUMPTION_PER_GROUP_MBA_HOUR/consumptionPerGroupMbaHour-en-csv-2023-11-16-to-2026-11-16.zip


## 3. Validate the ZIP
Verify that it is a readable ZIP archive and contains CSV files.

In [4]:
print("Checking the downloaded ZIP...")

try:
    if not zipfile.is_zipfile(temp_path):
        raise ValueError("Download is not a ZIP archive")

    with zipfile.ZipFile(temp_path) as archive:
        broken_file = archive.testzip()  # Check ZIP member CRCs.
        if broken_file:
            raise ValueError(f"Error in ZIP file: {broken_file}")
        csv_files = [
            item.filename for item in archive.infolist()
            if not item.is_dir() and item.filename.lower().endswith(".csv")
        ]

    if not csv_files:
        raise ValueError("ZIP contains no CSV files")
except Exception:
    temp_path.unlink(missing_ok=True)
    raise

print("ZIP is valid.")
print("CSV files found:", len(csv_files))
for name in csv_files[:10]:
    print("  -", name)
if len(csv_files) > 10:
    print(f"  ... and {len(csv_files) - 10} more")

Checking the downloaded ZIP...
ZIP is valid.
CSV files found: 1
  - part-00000-ff7fca7f-6cd0-4002-9a46-a778aeb5ed13-c000.csv


## 4. Save a version using SHA-256
A SHA-256 hash is a fingerprint of the file contents. If Elhub sends exactly the same ZIP again.

In [5]:
print("Calculating file fingerprint (SHA-256)...")
sha256 = hashlib.sha256()
with temp_path.open("rb") as file:
    for chunk in iter(lambda: file.read(1024 * 1024), b""):
        sha256.update(chunk)
checksum = sha256.hexdigest()

archive_path = BRONZE_DIR / f"{DATASET}_{checksum[:16]}.zip"

if archive_path.exists():
    # Check the full hash too.
    existing_hash = hashlib.sha256()
    with archive_path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            existing_hash.update(chunk)

    # Compare the full hash to ensure the existing archive is identical.        
    if existing_hash.hexdigest() != checksum:
        temp_path.unlink(missing_ok=True)
        raise ValueError("Filename collision or existing archive was modified")
    temp_path.unlink(missing_ok=True)
    status = "unchanged"
    print("Same archive already exists. No duplicates are saved.")
else:
    temp_path.replace(archive_path)
    status = "downloaded"
    print("New archive saved")

print("Status:", status)
print("SHA-256:", checksum)
print("File:", archive_path)

Calculating file fingerprint (SHA-256)...
New archive saved
Status: downloaded
SHA-256: 7c043bbaa2d599cceee7a234d3c3093713362b22883fc25642ed07d80416ed14
File: /home/roland/Desktop/DAT535_DEV2/group14/data/bronze/elhub/CONSUMPTION_PER_GROUP_MBA_HOUR/CONSUMPTION_PER_GROUP_MBA_HOUR_7c043bbaa2d599cc.zip


## 5. Save metadata and a run log
Metadata describes the archive version. The run log describes this execution.

In [6]:
original_name = unquote(Path(urlparse(final_url).path).name)
match = re.search(r"(\d{4}-\d{2}-\d{2})-to-(\d{4}-\d{2}-\d{2})", original_name)
period_start = match.group(1) if match else None
period_end = match.group(2) if match else None

metadata_path = BRONZE_DIR / f"{archive_path.name}.metadata.json"
run_dir = BRONZE_DIR / "_runs"
run_dir.mkdir(exist_ok=True)
run_path = run_dir / f"{run_id}.json"

# Create metadata for the downloaded archive.
metadata = {
    "dataset": DATASET,
    "source_url": SOURCE_URL,
    "final_url": final_url,
    "original_filename": original_name,
    "saved_filename": archive_path.name,
    "sha256": checksum,
    "size_bytes": archive_path.stat().st_size,
    "csv_files": csv_files,
    "advertised_period_start": period_start,
    "advertised_period_end": period_end,
    "period_verified": False,
    "first_seen_utc": datetime.now(timezone.utc).isoformat(),
}

# Do not overwrite the original metadata when the same version is downloaded again.
if not metadata_path.exists():
    metadata_path.write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")
    print("Created metadata:", metadata_path)
else:
    print("Metadata already exists; preserving it:", metadata_path)

# Create a run log
run_log = {
    "run_id": run_id,
    "started_at_utc": started_at,
    "finished_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset": DATASET,
    "status": status,
    "archive_path": str(archive_path),
    "sha256": checksum,
}
run_path.write_text(json.dumps(run_log, indent=2) + "\n", encoding="utf-8")

print("Saved run log:", run_path)
print("Advertised period:", period_start, "to", period_end)
print("Reminder: period has NOT been checked against CSV contents.")

Created metadata: /home/roland/Desktop/DAT535_DEV2/group14/data/bronze/elhub/CONSUMPTION_PER_GROUP_MBA_HOUR/CONSUMPTION_PER_GROUP_MBA_HOUR_7c043bbaa2d599cc.zip.metadata.json
Saved run log: /home/roland/Desktop/DAT535_DEV2/group14/data/bronze/elhub/CONSUMPTION_PER_GROUP_MBA_HOUR/_runs/98eb462bce0f.json
Advertised period: 2023-11-16 to 2026-11-16
Reminder: period has NOT been checked against CSV contents.


## 6. Inspect what Bronze contains
Print the saved metadata and list the files..

In [7]:
print("\n=== Saved archive ===")
print("Exists:", archive_path.exists())
print("Size (MiB):", round(archive_path.stat().st_size / (1024 * 1024), 2))

print("\n=== Metadata ===")
print(metadata_path.read_text(encoding="utf-8"))

print("=== Files in dataset Bronze folder ===")
for path in sorted(BRONZE_DIR.iterdir()):
    print("  ", path.name)

print("\nDone. Raw ZIP is stored in Bronze.")


=== Saved archive ===
Exists: True
Size (MiB): 10.79

=== Metadata ===
{
  "dataset": "CONSUMPTION_PER_GROUP_MBA_HOUR",
  "source_url": "https://api.elhub.no/energy-data/v0/price-areas.csv?dataset=CONSUMPTION_PER_GROUP_MBA_HOUR",
  "final_url": "https://frzq0sxltynr.objectstorage.eu-frankfurt-1.oci.customer-oci.com/p/hsnBS_rZZzT8AyoMxky1hRkvgrGCggmqDoCPntmrmKrtTrJfmkvpQpRgtct_3UBW/n/frzq0sxltynr/b/elhub-prod1-dataflow-dataset-osb/o/CONSUMPTION_PER_GROUP_MBA_HOUR/consumptionPerGroupMbaHour-en-csv-2023-11-16-to-2026-11-16.zip",
  "original_filename": "consumptionPerGroupMbaHour-en-csv-2023-11-16-to-2026-11-16.zip",
  "saved_filename": "CONSUMPTION_PER_GROUP_MBA_HOUR_7c043bbaa2d599cc.zip",
  "sha256": "7c043bbaa2d599cceee7a234d3c3093713362b22883fc25642ed07d80416ed14",
  "size_bytes": 11313160,
  "csv_files": [
    "part-00000-ff7fca7f-6cd0-4002-9a46-a778aeb5ed13-c000.csv"
  ],
  "advertised_period_start": "2023-11-16",
  "advertised_period_end": "2026-11-16",
  "period_verified": false,


## 7. Preview CSV columns without extracting the archive

Read only the header from the first CSV in the ZIP. The original ZIP remains unchanged in Bronze.


In [8]:
import pandas as pd
import zipfile

with zipfile.ZipFile(archive_path) as archive:
    with archive.open(csv_files[0]) as file:
        df = pd.read_csv(file, sep=None, engine="python", encoding="utf-8-sig")

print("Columns:", df.columns.tolist())
print(df.head())


Columns: ['START_TIME', 'END_TIME', 'PRICE_AREA', 'EIC', 'CONSUMPTION_GROUP', 'QUANTITY_KWH', 'METERING_POINT_COUNT', 'LAST_UPDATE_TIME']
                      START_TIME                       END_TIME PRICE_AREA  \
0  2023-11-16T00:00:00.000+01:00  2023-11-16T01:00:00.000+01:00        NO1   
1  2023-11-16T01:00:00.000+01:00  2023-11-16T02:00:00.000+01:00        NO1   
2  2023-11-16T02:00:00.000+01:00  2023-11-16T03:00:00.000+01:00        NO1   
3  2023-11-16T03:00:00.000+01:00  2023-11-16T04:00:00.000+01:00        NO1   
4  2023-11-16T04:00:00.000+01:00  2023-11-16T05:00:00.000+01:00        NO1   

                EIC CONSUMPTION_GROUP  QUANTITY_KWH  METERING_POINT_COUNT  \
0  10YNO-1--------2             cabin    133877.676                108651   
1  10YNO-1--------2             cabin    134589.250                108651   
2  10YNO-1--------2             cabin    135142.916                108651   
3  10YNO-1--------2             cabin    135542.558                108651   
4  10YNO